# deepSSF baseline demo

This notebook is a minimal demo of the MoveBench deepSSF baseline APIs.

Legacy exploratory work is preserved in `notebooks/deepssf_legacy.ipynb`.


In [1]:
import torch
from torch.utils.data import DataLoader

from movebench import Study
from movebench.models.deepssf import DeepSSF
from movebench.train import collate_deepssf, eval_deepssf, fit_deepssf


In [2]:
study_id = 'tsavo_lions'  # e.g. Study.list_available()[0]
layers_config = '../configs/layers_local.yaml'
device = torch.device('cpu')


In [3]:
study = Study(study_id)

# # Ensure rasters exist (download if missing), then extract a parametric point store
# # and compute normalization stats. deepSSF uses those stats via ChipDataset.
# try:
#     rep = study.extract_point_features(
#         config=layers_config,
#         splits=['train', 'test'],
#         parametric=True,
#         num_candidates=100,
#         random_seed=0,
#         n_workers=16,
#         skip_existing=True,
#         step_multiplier=2.0,
#         step_q=0.95,
#         gap_q=0.999,
#         bounds_buffer_deg=0.01,
#     )
# except FileNotFoundError as e:
#     if 'rasters' not in str(e):
#         raise
#     study.download_rasters(
#         config=layers_config,
#         splits=['train', 'test'],
#         n_workers=16,
#         skip_existing=True,
#     )
#     rep = study.extract_point_features(
#         config=layers_config,
#         splits=['train', 'test'],
#         parametric=True,
#         num_candidates=100,
#         random_seed=0,
#         n_workers=16,
#         skip_existing=True,
#         step_multiplier=2.0,
#         step_q=0.95,
#         gap_q=0.999,
#         bounds_buffer_deg=0.01,
#     )

# sampler_hash = rep['sampler_hash']
# study.compute_normalization_vals(
#     from_='points',
#     config=layers_config,
#     sampler_hash=sampler_hash,
#     split='train',
#     skip_existing=True,
# )


In [4]:
train_ds = study.deepssf_dataset(
    split='train',
    config=layers_config,
    # normalization_sampler_hash=sampler_hash,
    output_hw=(101, 101),
)
test_ds = study.deepssf_dataset(
    split='test',
    config=layers_config,
    # normalization_sampler_hash=sampler_hash,
    output_hw=(101, 101),
)

train_loader = DataLoader(
    train_ds, batch_size=16, shuffle=True, 
    num_workers=16, 
    collate_fn=collate_deepssf
)
test_loader = DataLoader(
    test_ds, batch_size=16, shuffle=False, 
    num_workers=16, 
    collate_fn=collate_deepssf
)


/Users/justinkay/miniconda3/envs/movebench/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
pixel_size_m = train_ds.pixel_size_m_estimate  # meters/pixel in EPSG:3857
pixel_size_m

399.47530773927764

In [ ]:
x1, x2, x3, y = next(iter(train_loader))
env_channels = int(x1.shape[1])
image_dim = int(x1.shape[-1])

model = DeepSSF(env_channels, 
                image_dim=image_dim, 
                pixel_size_m=pixel_size_m, 
                device=device)

fit = fit_deepssf(
    model,
    train_loader,
    test_loader,
    device,
    lr_movement=1e-5,
    lr_habitat=1e-4,
    weight_decay=0.0,
    max_steps=2000,
    eval_interval=10,
    patience=50,
    seed=0,
)

train_metrics = eval_deepssf(model, train_loader, device)
test_metrics = eval_deepssf(model, test_loader, device)

fit, train_metrics, test_metrics
